# Data

## CFB API Data

### Recruiting Rankings

In [18]:
import requests 
import pandas as pd
import os
import time
import ast
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

In [ ]:
os.makedirs("data", exist_ok=True)
api_key = "dtZVj1XqHVj5dfXxdc/nqukfZgSAP6wvRvPIHIMWpSTqSrMzmfzoxVJeCdMi2/Jo"
headers = {"Authorization": f"Bearer {api_key}"}
url = "https://api.collegefootballdata.com/recruiting/teams"

def pull_yearly(url, start=2005, end=2024, label=""):
    dfs = []
    for year in range(start, end):
        r = requests.get(url, headers=headers, params={"year": year})
        data = r.json()
        if isinstance(data, list) and len(data) > 0:
            dfs.append(pd.DataFrame(data))
        else:
            print(f"  No data for year {year}")
        time.sleep(0.2)
    result = pd.concat(dfs, ignore_index=True)
    print(f"{label}: {result.shape}")
    return result

# recruiting data
print("Pulling recruiting...")
recruiting_df = pull_yearly(
    "https://api.collegefootballdata.com/recruiting/teams",
    label="Recruiting"
)
recruiting_df.to_csv("data/recruiting.csv", index=False)

# SP+ ratings data
print("Pulling SP+ ratings...")
sp_df = pull_yearly(
    "https://api.collegefootballdata.com/ratings/sp",
    label="SP+"
)
sp_df.to_csv("data/sp_ratings.csv", index=False)

# win/loss records data
print("Pulling records...")
records_df = pull_yearly(
    "https://api.collegefootballdata.com/records",
    label="Records"
)
records_df.to_csv("data/records.csv", index=False)

# caoches data
print("Pulling coaches...")
coaches_df = pull_yearly(
    "https://api.collegefootballdata.com/coaches",
    label="Coaches"
)
coaches_df.to_csv("data/coaches.csv", index=False)

print("\nAll done — files saved to data/")

Pulling recruiting...
Recruiting: (3388, 4)
Pulling SP+ ratings...


/var/folders/8s/x70pwprd6p34_2y8q6xy3mk80000gn/T/ipykernel_80068/332811132.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat(dfs, ignore_index=True)


SP+: (2401, 10)
Pulling records...
Records: (6140, 14)
Pulling talent...
  No data for year 2005
  No data for year 2006
  No data for year 2007
  No data for year 2008
  No data for year 2009
  No data for year 2010
  No data for year 2011
  No data for year 2012
  No data for year 2013
  No data for year 2014
Talent: (2010, 3)
Pulling coaches...
Coaches: (2493, 4)

All done — files saved to data/


## Checking Data

In [9]:
files = {
    "recruiting": "data/recruiting.csv",
    "sp_ratings": "data/sp_ratings.csv",
    "records": "data/records.csv",
    "coaches": "data/coaches.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)
    print(f"{name.upper()}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    print(df.head(3).to_string())
    print()

RECRUITING
Shape: (3388, 4)
Columns: ['year', 'team', 'rank', 'points']
Missing values:
Series([], dtype: int64)

Sample rows:
   year           team  rank  points
0  2005            USC     1  285.12
1  2005  Florida State     2  281.52
2  2005       Oklahoma     3  279.20

SP_RATINGS
Shape: (2401, 10)
Columns: ['year', 'team', 'conference', 'rating', 'ranking', 'secondOrderWins', 'sos', 'offense', 'defense', 'specialTeams']
Missing values:
conference          19
ranking             19
secondOrderWins    659
sos                659
dtype: int64

Sample rows:
   year        team conference  rating  ranking  secondOrderWins    sos                                                                                                                                                                                      offense                                                                                                                                                                               

Recruiting Data: clean, no issues so far
SP+: the offense, defense, and specialTeams columns are nested as strings
Records: total, conferenceGames... are all nested dicts. Need to parse total. Also includes FCS teams so I'll have to filter to classification == 'fbs'
Coaches: seasons column is a nested list of dicts

In [15]:
coaches_raw = pd.read_csv("data/coaches.csv")

# Check what the seasons column actually looks like
print(type(coaches_raw["seasons"].iloc[0]))
print()
print(repr(coaches_raw["seasons"].iloc[0]))

<class 'str'>

"[{'school': 'Wisconsin', 'year': 2005, 'games': 13, 'wins': 10, 'losses': 3, 'ties': 0, 'preseasonRank': None, 'postseasonRank': 15, 'srs': 13.2, 'spOverall': 11.1, 'spOffense': 35.3, 'spDefense': 24.2}]"


In [20]:
# recruiting (already clean)
recruiting = pd.read_csv("data/recruiting.csv")

# SP+ (extracting only what is needed)
sp = pd.read_csv("data/sp_ratings.csv")
sp = sp[["year", "team", "conference", "rating", "ranking"]].copy()
sp.rename(columns={"rating": "sp_rating", "ranking": "sp_ranking"}, inplace=True)

# records (parse nested total dict, filter to FBS) 
records_raw = pd.read_csv("data/records.csv")
records_raw = records_raw[records_raw["classification"] == "fbs"].copy()

def parse_total(val):
    try:
        d = ast.literal_eval(val)
        return d.get("wins"), d.get("losses"), d.get("games")
    except:
        return None, None, None

records_raw[["wins", "losses", "games"]] = records_raw["total"].apply(
    lambda x: pd.Series(parse_total(x))
)
records = records_raw[["year", "team", "conference", "wins", "losses", "games"]].copy()

# talent (already clean)
talent = pd.read_csv("data/talent.csv")

# coaches (explode seasons list into one row per coach-season)
coaches_raw = pd.read_csv("data/coaches.csv")

rows = []
failed = 0
for _, row in coaches_raw.iterrows():
    try:
        seasons = ast.literal_eval(row["seasons"])
        for s in seasons:
            rows.append({
                "first_name": row["firstName"],
                "last_name": row["lastName"],
                "school": s.get("school"),
                "year": s.get("year"),
                "wins": s.get("wins"),
                "losses": s.get("losses"),
                "sp_overall": s.get("spOverall"),
                "sp_offense": s.get("spOffense"),
                "sp_defense": s.get("spDefense"),
            })
    except:
        continue

coaches = pd.DataFrame(rows)
coaches["coach_name"] = coaches["first_name"] + " " + coaches["last_name"]

# check
for name, df in [("recruiting", recruiting), ("sp", sp), 
                  ("records", records), ("talent", talent), 
                  ("coaches", coaches)]:
    print(f"{name}: {df.shape}")
    print(df.head(2).to_string())
    print()

# saving cleaned versions of data
recruiting.to_csv("data/recruiting_clean.csv", index=False)
sp.to_csv("data/sp_clean.csv", index=False)
records.to_csv("data/records_clean.csv", index=False)
talent.to_csv("data/talent_clean.csv", index=False)
coaches.to_csv("data/coaches_clean.csv", index=False)

print("All cleaned files saved.")

recruiting: (3388, 4)
   year           team  rank  points
0  2005            USC     1  285.12
1  2005  Florida State     2  281.52

sp: (2401, 5)
   year   team conference  sp_rating  sp_ranking
0  2005  Texas        SEC       35.3         1.0
1  2005    USC    Big Ten       34.7         2.0

records: (2383, 6)
   year    team      conference  wins  losses  games
0  2005  Auburn             SEC     9       3     12
1  2005     UAB  Conference USA     5       6     11

talent: (2010, 3)
   year     team  talent
0  2015  Alabama  981.90
1  2015      USC  926.71

coaches: (2495, 10)
  first_name last_name     school  year  wins  losses  sp_overall  sp_offense  sp_defense     coach_name
0      Barry   Alvarez  Wisconsin  2005    10       3        11.1        35.3        24.2  Barry Alvarez
1      Chuck     Amato   NC State  2005     7       5         9.4        22.1        12.7    Chuck Amato

All cleaned files saved.


### School Name Standardization

In [21]:


# load cleaned data
recruiting = pd.read_csv("data/recruiting_clean.csv")
draft = pd.read_csv("data/nfl_draft_prospects.csv")

# Filter draft to 2005+ and actual drafted players
draft = draft[(draft["draft_year"] >= 2005) & (draft["round"].notna())].copy()

# Get unique school names from each
cfbd_teams = set(recruiting["team"].dropna().str.strip().unique())
draft_schools = set(draft["school"].dropna().str.strip().unique())

# Find mismatches
only_in_draft = sorted(draft_schools - cfbd_teams)
only_in_cfbd = sorted(cfbd_teams - draft_schools)
in_both = sorted(draft_schools & cfbd_teams)

print(f"Schools in both: {len(in_both)}")
print(f"Schools only in draft (need mapping): {len(only_in_draft)}")
print(f"Schools only in CFBD (no draft picks or name diff): {len(only_in_cfbd)}")
print()
print("=== DRAFT SCHOOLS NOT IN CFBD ===")
for s in only_in_draft:
    print(f"  '{s}'")
print()
print("=== CFBD SCHOOLS NOT IN DRAFT ===")
for s in only_in_cfbd:
    print(f"  '{s}'")

Schools in both: 205
Schools only in draft (need mapping): 87
Schools only in CFBD (no draft picks or name diff): 50

=== DRAFT SCHOOLS NOT IN CFBD ===
  'Abilene Christian'
  'Albany'
  'Albany State (GA)'
  'Albion'
  'Appalachian State'
  'Ashland'
  'Australia'
  'Bentley'
  'Bethel (TN)'
  'Bloomsburg'
  'California (PA)'
  'Central Missouri'
  'Central Missouri State'
  'Chadron State'
  'Charleston (WV)'
  'Colorado State-Pueblo'
  'Concordia-St. Paul'
  'Connecticut'
  'East Central'
  'Ferris State'
  'Florida Intl'
  'Fort Hays State'
  'Grambling State'
  'Grand Valley State'
  'Harding'
  'Hillsdale'
  'Hobart'
  'Hofstra'
  'Humboldt State'
  'Indiana (PA)'
  'Kutztown'
  'Lane'
  'Lenoir-Rhyne'
  'Lindenwood'
  'Louisiana Monroe'
  'Manitoba'
  'Mars Hill'
  'McGill'
  'McNeese State'
  'Miami (FL)'
  'Michigan Tech'
  'Midwestern State'
  'Missouri Southern State'
  'Missouri Western'
  'Morehouse'
  'Mount Union'
  'Nebraska-Omaha'
  'Newberry'
  'Nicholls State'
  'Nor

Most mismatches are abbreviations, name variations, and FCS schools 

In [ ]:
# Schools in draft that need mapping to CFBD names
# drop no fbs teams
draft_to_cfbd = {
    # fbs name fixes
    "Appalachian State":        "App State",
    "Connecticut":              "UConn",
    "Florida Intl":             "FIU",
    "Louisiana Monroe":         "UL Monroe",
    "San Jose State":           "San José State",
    "Southern Mississippi":     "Southern Miss",
    "Stephen F Austin":         "Stephen F. Austin",
    "UMass":                    "Massachusetts",
    "UT San Antonio":           "UTSA",
    "Tennessee-Martin":         "UT Martin",
    "Sam Houston State":        "Sam Houston",
    "North Carolina State":     "NC State",
    "Miami (FL)":               "Miami",
    "McNeese State":            "McNeese",
    "Nicholls State":           "Nicholls",
    "Prairie View":             "Prairie View A&M",
    "Presbyterian College":     "Presbyterian",
    "Southeastern Louisiana":   "SE Louisiana",

    # non fbs
    "Abilene Christian":        None,
    "Albany":                   None,
    "Albany State (GA)":        None,
    "Albion":                   None,
    "Ashland":                  None,
    "Australia":                None,
    "Bentley":                  None,
    "Bethel (TN)":              None,
    "Bloomsburg":               None,
    "California (PA)":          None,
    "Central Missouri":         None,
    "Central Missouri State":   None,
    "Chadron State":            None,
    "Charleston (WV)":          None,
    "Colorado State-Pueblo":    None,
    "Concordia-St. Paul":       None,
    "East Central":             None,
    "Ferris State":             None,
    "Fort Hays State":          None,
    "Grambling State":          None,
    "Grand Valley State":       None,
    "Harding":                  None,
    "Hillsdale":                None,
    "Hobart":                   None,
    "Hofstra":                  None,
    "Humboldt State":           None,
    "Indiana (PA)":             None,
    "Kutztown":                 None,
    "Lane":                     None,
    "Lenoir-Rhyne":             None,
    "Lindenwood":               None,
    "Manitoba":                 None,
    "Mars Hill":                None,
    "McGill":                   None,
    "Michigan Tech":            None,
    "Midwestern State":         None,
    "Missouri Southern State":  None,
    "Missouri Western":         None,
    "Morehouse":                None,
    "Mount Union":              None,
    "Nebraska-Omaha":           None,
    "Newberry":                 None,
    "North Alabama":            None,
    "Northeastern State":       None,
    "Northwest Missouri State": None,
    "Pittsburg State":          None,
    "Saginaw Valley":           None,
    "Sioux Falls":              None,
    "St. Augustine's":          None,
    "St. John's (MN)":          None,
    "St. Paul's College":       None,
    "Stillman":                 None,
    "Tarleton State":           None,
    "Tuskegee":                 None,
    "UW-Whitewater":            None,
    "Valdosta State":           None,
    "Virginia State":           None,
    "Washburn":                 None,
    "West Alabama":             None,
    "West Georgia":             None,
    "West Texas A&M":           None,
    "Western Ontario":          None,
    "Western Oregon":           None,
    "Wheaton College (Ill)":    None,
    "Whitworth":                None,
    "William Penn":             None,
    "Wingate":                  None,
    "Winston-Salem":            None,
    "Wisconsin-Whitewater":     None,
}

# apply mapping to draft data
draft["school_std"] = draft["school"].map(
    lambda x: draft_to_cfbd.get(x, x) if x in draft_to_cfbd else x
)

# drop rows where mapping is None (FCS/D2/international)
draft_fbs = draft[draft["school_std"].notna()].copy()

# verify overlap
cfbd_teams = set(recruiting["team"].dropna().str.strip().unique())
draft_schools_std = set(draft_fbs["school_std"].dropna().str.strip().unique())

still_missing = sorted(draft_schools_std - cfbd_teams)
matched = sorted(draft_schools_std & cfbd_teams)

print(f"Matched schools: {len(matched)}")
print(f"Still unmatched after mapping: {len(still_missing)}")
if still_missing:
    print("\nStill unmatched:")
    for s in still_missing:
        print(f"  '{s}'")

print(f"\nDraft picks remaining after FCS drop: {draft_fbs.shape[0]}")
print(f"Draft picks dropped (FCS/D2): {draft.shape[0] - draft_fbs.shape[0]}")

Matched schools: 216
Still unmatched after mapping: 1

Still unmatched:
  'FIU'

Draft picks remaining after FCS drop: 4221
Draft picks dropped (FCS/D2): 110


Checking what CFBD calls the 4 schools not standardized

In [23]:
search_terms = ["FIU", "Florida International"]

for term in search_terms:
    matches = [t for t in cfbd_teams if term.lower() in t.lower()]
    if matches:
        print(f"Search '{term}': {matches}")

Search 'Florida International': ['Florida International']


In [ ]:
# fix the 1 remaining mismatch
draft_to_cfbd["Florida Intl"] = "Florida International"  # update existing entry

# Re-apply mapping
draft["school_std"] = draft["school"].map(
    lambda x: draft_to_cfbd.get(x, x) if x in draft_to_cfbd else x
)

# drop None mappings
draft_fbs = draft[draft["school_std"].notna()].copy()

# verify
cfbd_teams = set(recruiting["team"].dropna().str.strip().unique())
draft_schools_std = set(draft_fbs["school_std"].dropna().str.strip().unique())

still_missing = sorted(draft_schools_std - cfbd_teams)
matched = sorted(draft_schools_std & cfbd_teams)

print(f"Matched schools: {len(matched)}")
print(f"Still unmatched: {len(still_missing)}")
print(f"Draft picks remaining: {draft_fbs.shape[0]}")

Matched schools: 216
Still unmatched: 0
Draft picks remaining: 4221


Save clean draft file

In [25]:
draft_fbs.to_csv("data/draft_clean.csv", index=False)
print("Saved draft_clean.csv")
print(draft_fbs[["draft_year", "school_std", "round", "overall", "player_name", "position"]].head(5).to_string())

Saved draft_clean.csv
      draft_year school_std  round  overall       player_name       position
7599        2005       Utah    1.0      1.0        Alex Smith    Quarterback
7600        2005     Auburn    1.0      2.0      Ronnie Brown   Running Back
7601        2005   Michigan    1.0      3.0   Braylon Edwards  Wide Receiver
7602        2005      Texas    1.0      4.0     Cedric Benson   Running Back
7603        2005     Auburn    1.0      5.0  Carnell Williams   Running Back


### Weighted Draft Score
Giving a weighted raft score:
Round 1 pick = 7
Round 2 = 6
Round 3 = 5
Round 4 = 4
Round 5 = 3
Round 6 = 2
Round 7 = 1

In [26]:
draft = pd.read_csv("data/draft_clean.csv")

# round weight
round_weights = {1: 7, 2: 5, 3: 4, 4: 3, 5: 2, 6: 1, 7: 1}

draft["round"] = draft["round"].astype(int)
draft["weight"] = draft["round"].map(round_weights)

# collapse to school year level
draft_scored = (
    draft
    .groupby(["draft_year", "school_std"])
    .agg(
        total_picks=("player_name", "count"),
        weighted_draft_score=("weight", "sum"),
        first_round_picks=("round", lambda x: (x == 1).sum()),
        second_round_picks=("round", lambda x: (x == 2).sum()),
    )
    .reset_index()
    .rename(columns={"draft_year": "year", "school_std": "team"})
)

# add zero rows for schools with no draft picks that year, every FBS school should appear every year even with 0 picks
all_years = range(draft["draft_year"].min(), draft["draft_year"].max() + 1)
all_teams = draft["school_std"].unique()

full_index = pd.MultiIndex.from_product(
    [all_years, all_teams], names=["year", "team"]
)
draft_scored = (
    draft_scored
    .set_index(["year", "team"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

print(draft_scored.shape)
print(draft_scored.head(10).to_string())
print()

# checking alabama data (should have lots of picks and high weighted draft score)
print("Alabama draft scores by year:")
print(draft_scored[draft_scored["team"] == "Alabama"].to_string())

(3672, 6)
   year            team  total_picks  weighted_draft_score  first_round_picks  second_round_picks
0  2005            Utah            5                    14                  1                   0
1  2005          Auburn            5                    29                  4                   0
2  2005        Michigan            3                    19                  2                   1
3  2005           Texas            3                    15                  2                   0
4  2005   West Virginia            3                    13                  1                   0
5  2005  South Carolina            3                     9                  1                   0
6  2005           Miami            5                    21                  1                   1
7  2005             USC            5                    25                  2                   2
8  2005            Troy            1                     7                  1                   0
9  2005   

In [27]:
draft_scored.to_csv("data/draft_scored.csv", index=False)


## Lagging

Creating lags of 3-5 years after a recruiting class to capture that recruiting class' draft results. Typically, players will play 3-5 years before entering the draft. 3 is the minimum amount of time a player must be enrolled in college before being eligible for the draft, and 5 years after captures players who redshirted their freshman year and then played 4 years. This will allow for better analysis of understanding a specific year's recruit to draft outcomes.

In [29]:
recruiting = pd.read_csv("data/recruiting_clean.csv")
draft_scored = pd.read_csv("data/draft_scored.csv")

lag_rows = []

for _, rec_row in recruiting.iterrows():
    rec_year = rec_row["year"]
    team = rec_row["team"]
    
    # +3 to +5 year window
    draft_window = draft_scored[
        (draft_scored["team"] == team) &
        (draft_scored["year"] >= rec_year + 3) &
        (draft_scored["year"] <= rec_year + 5)
    ]
    
    lag_rows.append({
        "year": rec_year,
        "team": team,
        "rec_rank": rec_row["rank"],
        "rec_points": rec_row["points"],
        "draft_picks_total": draft_window["total_picks"].sum(),
        "draft_score_total": draft_window["weighted_draft_score"].sum(),
        "first_round_total": draft_window["first_round_picks"].sum(),
        "draft_years_covered": len(draft_window),
    })

lagged_df = pd.DataFrame(lag_rows)

# complete window = all 3 draft years present (+3, +4, +5) (this will eliminate tail ends of the data)
lagged_df["complete_window"] = lagged_df["draft_years_covered"] == 3

print(lagged_df.shape)
print()
print(f"Complete windows: {lagged_df['complete_window'].sum()}")
print(f"Incomplete windows: {(~lagged_df['complete_window']).sum()}")
print()
print("Alabama lagged scores:")
print(lagged_df[lagged_df["team"] == "Alabama"].to_string())

(3388, 9)

Complete windows: 1838
Incomplete windows: 1550

Alabama lagged scores:
      year     team  rec_rank  rec_points  draft_picks_total  draft_score_total  first_round_total  draft_years_covered  complete_window
15    2005  Alabama        16      216.52                 11                 49                  3                    3             True
130   2006  Alabama        13      233.66                 16                 78                  7                    3             True
250   2007  Alabama        12      241.78                 20                 97                 10                    3             True
357   2008  Alabama         3      291.61                 22                104                 11                    3             True
475   2009  Alabama         3      288.63                 25                104                  9                    3             True
594   2010  Alabama         4      284.20                 24                 91                

In [30]:
lagged_df.to_csv("data/lagged_df.csv", index=False)
print("Saved lagged_df.csv")

Saved lagged_df.csv


In [31]:


# complete windows only for clean analysis
complete = lagged_df[lagged_df["complete_window"] == True].copy()

print(f"Working with {complete.shape[0]} complete recruiting class observations")
print(f"Covering {complete['team'].nunique()} unique programs")
print(f"Year range: {complete['year'].min()} - {complete['year'].max()}")
print()



Working with 1838 complete recruiting class observations
Covering 214 unique programs
Year range: 2005 - 2016



### Building Development Residual

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# complete windows only
complete = lagged_df[lagged_df["complete_window"] == True].copy()

# fit regression: draft_score ~ rec_points
X = complete[["rec_points"]].values
y = complete["draft_score_total"].values

model = LinearRegression()
model.fit(X, y)

# residuals = actual draft score - expected draft score given recruiting
complete["expected_draft_score"] = model.predict(X)
complete["development_residual"] = complete["draft_score_total"] - complete["expected_draft_score"]

print(f"R² of recruiting = draft score: {model.score(X, y):.3f}")
print()
print(complete[["year", "team", "rec_points", "draft_score_total", 
               "expected_draft_score", "development_residual"]].head(10).to_string())

# save
complete.to_csv("data/complete_lagged.csv", index=False)
print("\nSaved complete_lagged.csv")

R² of recruiting → draft score: 0.506

   year           team  rec_points  draft_score_total  expected_draft_score  development_residual
0  2005            USC      285.12                118             41.619445             76.380555
1  2005  Florida State      281.52                 21             40.979293            -19.979293
2  2005       Oklahoma      279.20                 64             40.566750             23.433250
3  2005      Tennessee      276.14                 46             40.022621              5.977379
4  2005       Michigan      266.54                 36             38.315549             -2.315549
5  2005           Iowa      252.89                 44             35.888306              8.111694
6  2005        Georgia      248.26                 44             35.064999              8.935001
7  2005       Nebraska      247.47                 21             34.924521            -13.924521
8  2005          Miami      246.36                 28             34.727141    

## Merging datasets into master dataset

In [33]:

# load all cleaned datasets
complete = pd.read_csv("data/complete_lagged.csv")
sp = pd.read_csv("data/sp_clean.csv")
records = pd.read_csv("data/records_clean.csv")
talent = pd.read_csv("data/talent_clean.csv")
coaches = pd.read_csv("data/coaches_clean.csv")

# base is complete lagged df — join everything on team + year
master = complete.copy()

# SP+ ratings
master = master.merge(
    sp[["year", "team", "sp_rating", "sp_ranking", "conference"]],
    on=["year", "team"],
    how="left"
)

# Win/loss records
master = master.merge(
    records[["year", "team", "wins", "losses", "games"]],
    on=["year", "team"],
    how="left"
)

# Coaches
coaches_slim = coaches[["school", "year", "coach_name", "sp_overall"]].rename(
    columns={"school": "team", "sp_overall": "coach_sp_overall"}
)
master = master.merge(
    coaches_slim,
    on=["year", "team"],
    how="left"
)

# Win percentage
master["win_pct"] = master["wins"] / master["games"]

print(master.shape)
print(master.columns.tolist())
print()
print(master.head(3).to_string())
print()
print("Missing values:")
print(master.isnull().sum()[master.isnull().sum() > 0])

master.to_csv("data/master.csv", index=False)
print("\nSaved master.csv")

(1875, 20)
['year', 'team', 'rec_rank', 'rec_points', 'draft_picks_total', 'draft_score_total', 'first_round_total', 'draft_years_covered', 'complete_window', 'expected_draft_score', 'development_residual', 'sp_rating', 'sp_ranking', 'conference', 'wins', 'losses', 'games', 'coach_name', 'coach_sp_overall', 'win_pct']

   year           team  rec_rank  rec_points  draft_picks_total  draft_score_total  first_round_total  draft_years_covered  complete_window  expected_draft_score  development_residual  sp_rating  sp_ranking conference  wins  losses  games    coach_name  coach_sp_overall   win_pct
0  2005            USC         1      285.12                 28                118                  7                    3             True             41.619445             76.380555       34.7         2.0    Big Ten  12.0     1.0   13.0  Pete Carroll              34.7  0.923077
1  2005  Florida State         2      281.52                  7                 21                  1                